In [148]:
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

torch.manual_seed(40)

In [149]:
df = pd.read_csv('../3. Dataset/Dataset_English_Hindi.csv')
df.tail(3)

,English,Hindi
130473,"As for the other derivatives of sulphur , the ...","जहां तक गंधक के अन्य उत्पादों का प्रश्न है , द..."
130474,its complicated functioning is defined thus in...,Zरचना-प्रकिया को उसने एक पहेली में यों बांधा है .
130475,They've just won four government contracts to ...,हाल ही में उन्हें सरकारी ठेका मिला है करीब सौ ...


In [150]:
df.isna().sum()

English      2
Hindi      312
dtype: int64

In [151]:
df = df.dropna(subset=['English', 'Hindi']).reset_index(drop=True)

In [152]:
df.isna().sum()

English    0
Hindi      0
dtype: int64

In [153]:
def tokenize(text):
  
  text = text.lower()
  text = text.replace('?','')
  text = text.replace("'","")
  
  return text.split()

def build_vocab_source(sentences):
  
  vocab = {'<PAD>' : 0 , '<UNK>':1 }

  for sentence in sentences :
    tokenized_sentence = tokenize(sentence)
    for token in tokenized_sentence:
       if token not in vocab:     
            vocab[token] = len(vocab)
        
  return vocab


def build_vocab_target(sentences):
  
  vocab = {'<PAD>': 0, '<UNK>': 1, '<SOS>': 2, '<EOS>': 3}

  for sentence in sentences :
    tokenized_sentence = tokenize(sentence)
    for token in tokenized_sentence:
       if token not in vocab:     
            vocab[token] = len(vocab)
        
  return vocab


vocab_en = build_vocab_source(df['English'].values)
vocab_hi = build_vocab_target(df['Hindi'])

print(len(vocab_en) , len(vocab_hi))

100963 94947


In [154]:
def text_to_indices(text, vocab , num : int = 0):
  
  indexed_text = []

  if num == 1:
    indexed_text.append(vocab['<SOS>'])
  
  for token in tokenize(text):
    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])
    
  if num == 2:
    indexed_text.append(vocab['<EOS>'])

  return indexed_text

In [155]:
train_df , test_df = train_test_split(df , test_size=0.2  , random_state=20)

In [158]:
class TranslationDataset(Dataset):

    def __init__(self, df,src_vocab,tgt_vocab):
        
        self.df = df
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab

    def __len__(self):

        return self.df.shape[0]
    
    def __getitem__(self, idx):

        english_sentence_input = text_to_indices(self.df.iloc[idx]['English'], self.src_vocab)
        hindi_sentence_input = text_to_indices(self.df.iloc[idx]['Hindi'], self.tgt_vocab,1)
        hindi_sentence_target = text_to_indices(self.df.iloc[idx]['Hindi'], self.tgt_vocab,2)


        return torch.tensor(english_sentence_input), torch.tensor(hindi_sentence_input) ,torch.tensor(hindi_sentence_target) 


In [159]:
def collate_fn(batch):

    encoder_input, decoder_input , decoder_output = zip(*batch)
    
    encoder_input = pad_sequence(encoder_input, batch_first=True, padding_value=vocab_en['<PAD>'])
    decoder_input = pad_sequence(decoder_input, batch_first=True, padding_value=vocab_hi['<PAD>'])
    decoder_output = pad_sequence(decoder_output, batch_first=True, padding_value=vocab_hi['<PAD>'])

    # note : it consider max_len for each batch.

    return encoder_input, decoder_input , decoder_output


In [160]:
train_dataset = TranslationDataset(df,vocab_en,vocab_hi)
train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)

In [161]:
src_vocab_size = len(vocab_en)
tgt_vocab_size = len(vocab_hi)

src_seq_len = 512  # used to caculate positinal embeddings
tgt_seq_len = 512  # used to caculate positinal embeddings

d_model = 512 # embedding dim
N = 6 # no of layers 
h = 8 # no of heads (d_model % h == 0)
dropout = 0.1 
d_ff = 2048 # neurons in hidden layer


In [162]:
from models.Transformer_Architecture import build_transformer

model = build_transformer(src_vocab_size, tgt_vocab_size , src_seq_len , tgt_seq_len , d_model , N ,h , dropout , d_ff ) 

In [163]:
# check this how it works

def create_encoder_mask(encoder_input, pad_idx=0):
    # src: (batch, src_len)
    return (encoder_input != pad_idx).unsqueeze(1).unsqueeze(2)
    # (batch, 1, 1, src_len)

def subsequent_mask(size):
    return torch.tril(torch.ones(size, size)).bool()

def create_decoder_mask(decoder_input, pad_idx=0):

    batch, decoder_input_len = decoder_input.shape
    pad_mask = (decoder_input != pad_idx).unsqueeze(1).unsqueeze(2)
    causal_mask = subsequent_mask(decoder_input_len).to(decoder_input.device)
    return pad_mask & causal_mask


In [164]:
batch = next(iter(train_dataloader))

encoder_input, decoder_input , decoder_output = batch

encoder_input_mask = create_encoder_mask(encoder_input)
decoder_input_mask = create_decoder_mask(decoder_input)


print(encoder_input.shape , decoder_input.shape , encoder_input_mask.shape , decoder_input_mask.shape)


torch.Size([4, 14]) torch.Size([4, 17]) torch.Size([4, 1, 1, 14]) torch.Size([4, 1, 17, 17])


In [165]:
from torchinfo import summary

summary(
    model,
    input_data=(encoder_input,decoder_input,encoder_input_mask,decoder_input_mask)
)

Layer (type:depth-idx)                                  Output Shape              Param #
Transformer                                             [4, 17, 94947]            --
├─InputEmbeddings: 1-1                                  [4, 14, 512]              --
│    └─Embedding: 2-1                                   [4, 14, 512]              51,693,056
├─PositionalEncoding: 1-2                               [4, 14, 512]              --
│    └─Dropout: 2-2                                     [4, 14, 512]              --
├─Encoder: 1-3                                          [4, 14, 512]              --
│    └─ModuleList: 2-3                                  --                        --
│    │    └─EncoderBlock: 3-1                           [4, 14, 512]              3,150,336
│    │    └─EncoderBlock: 3-2                           [4, 14, 512]              3,150,336
│    │    └─EncoderBlock: 3-3                           [4, 14, 512]              3,150,336
│    │    └─EncoderBlock: 3-4  

In [166]:
learning_rate = 0.001
epochs = 5

criterion = torch.nn.CrossEntropyLoss(ignore_index=vocab_en['<PAD>'])
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# Training

In [167]:
from tqdm import tqdm

for epoch in range(epochs):
    
    total_loss = 0

    # tqdm wraps the dataloader
    for batch in tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{epochs}", leave=False):
        
        encoder_input, decoder_input , decoder_output = batch

        encoder_input_mask = create_encoder_mask(encoder_input)
        decoder_input_mask = create_decoder_mask(decoder_input)
        
        optimizer.zero_grad()

        output = model(encoder_input,decoder_input,encoder_input_mask,decoder_input_mask)

        b, t, v = output.shape
        loss = criterion(output.reshape(b*t, v), decoder_output.reshape(b*t))
        
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        break

    print(f"Epoch: {epoch+1}, Loss: {total_loss:.4f}")


Epoch: 1, Loss: 11.4740


Epoch: 2, Loss: 11.3468


Epoch: 3, Loss: 11.0699


Epoch: 4, Loss: 10.7742


Epoch: 5, Loss: 10.4169


# Testing 

# Prediction for single 

# Training on GPU

In [114]:
device = 'cpu'
if hasattr(torch,'mps') and torch.backends.mps.is_available():
    device = 'mps'
    print("MPS is available")

MPS is available


In [ ]:
model.to(device)

for epoch in range(epochs):
    
    total_loss = 0

    # tqdm wraps the dataloader
    for batch in tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{epochs}", leave=False):
        
        src, tgt = batch
        src = src.to(device)
        tgt = tgt.to(device)

        src_mask = create_src_mask(src).to(device)
        tgt_mask = create_tgt_mask(tgt).to(device)

        optimizer.zero_grad()

        output = model(src, tgt, src_mask, tgt_mask)

        b, t, v = output.shape
        loss = criterion(output.reshape(b*t, v), tgt.reshape(b*t))
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        break

    print(f"Epoch: {epoch+1}, Loss: {total_loss:.4f}")
